# Lecture 7: Financial Time Series
In this notebook we will cover:
- Statistical properties of financial returns
- ARCH and GARCH models (conditional heteroskedasticity)
- EGARCH — asymmetric volatility (leverage effect)
- Geometric Brownian Motion and Black-Scholes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# arch library
try:
    from arch import arch_model
    print("arch library available.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'arch', '-q'])
    from arch import arch_model
    print("arch library installed.")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Ready.")


## 7.1 Properties of Financial Returns

**Log return:** $r_t = \log(P_t/P_{t-1}) = \log P_t - \log P_{t-1}$

**Stylized Facts:**
1. **Heavy tails** — more extreme observations than implied by the normal distribution
2. **Volatility clustering** — periods of high volatility tend to cluster
3. **Leverage effect** — negative shocks increase volatility more than positive shocks
4. **Autocorrelation** — absent in returns, present in squared returns


In [ ]:
# ── Realistic Stock Price Simulation (GBM + GARCH volatility) ──

np.random.seed(42)
n = 1000

# GARCH(1,1) volatility process
omega, alpha, beta = 0.00001, 0.10, 0.85
sigma2 = np.zeros(n)
sigma2[0] = omega / (1 - alpha - beta)
eps = np.zeros(n)
r = np.zeros(n)
mu = 0.0003  # daily mean return

for t in range(1, n):
    eps[t] = np.random.normal(0, np.sqrt(sigma2[t-1]))
    r[t] = mu + eps[t]
    sigma2[t] = omega + alpha * eps[t]**2 + beta * sigma2[t-1]

# Price series
P = 100 * np.exp(np.cumsum(r))
sigma_t = np.sqrt(sigma2)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Price
axes[0,0].plot(P, color='steelblue', lw=0.8)
axes[0,0].set_title('Simulated Price Series', fontweight='bold')
axes[0,0].set_ylabel('Price')

# Returns
axes[0,1].plot(r, color='darkorange', lw=0.5, alpha=0.7)
axes[0,1].set_title('Log Returns $r_t$', fontweight='bold')

# Volatility
axes[1,0].plot(sigma_t * np.sqrt(252), color='red', lw=0.8)
axes[1,0].set_title('True Volatility (Annualised %)', fontweight='bold')

# Squared returns (volatility clustering)
axes[1,1].plot(r**2, color='purple', lw=0.5, alpha=0.8)
axes[1,1].set_title('$r_t^2$ — Volatility Clustering', fontweight='bold')

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(r, lags=20, ax=axes[2,0], title='ACF($r_t$) — No Autocorrelation')
plot_acf(r**2, lags=20, ax=axes[2,1], title='ACF($r_t^2$) — Autocorrelation Present!')

plt.tight_layout()
plt.show()

# Normality test
stat, p = stats.shapiro(r[:500])
print(f"Shapiro-Wilk p-value: {p:.6f} -> {'Non-normal' if p<0.05 else 'Normal'}")
kurt = stats.kurtosis(r)
print(f"Excess kurtosis: {kurt:.3f} (Normal=0, heavy tails>0)")


## 7.2 ARCH Model

**ARCH(q) — AutoRegressive Conditional Heteroskedasticity** (Engle, 1982):

$$r_t = \sigma_t \epsilon_t, \quad \epsilon_t \sim WN(0,1)$$
$$\sigma_t^2 = \omega + \alpha_1 r_{t-1}^2 + \cdots + \alpha_q r_{t-q}^2$$

**ARCH Effect Test:** Ljung-Box test on the $r_t^2$ series.


In [ ]:
# ── ARCH Effect Test ──

lb_r = acorr_ljungbox(r, lags=[5, 10, 20], return_df=True)
lb_r2 = acorr_ljungbox(r**2, lags=[5, 10, 20], return_df=True)

print("Ljung-Box Test — r_t (returns):")
print(lb_r[['lb_stat', 'lb_pvalue']].round(4))

print("\nLjung-Box Test — r_t^2 (squared returns):")
print(lb_r2[['lb_stat', 'lb_pvalue']].round(4))
print("\n=> p<0.05 for r_t^2: ARCH effect present!")


## 7.3 GARCH(1,1) Model

**GARCH(p,q) — Generalized ARCH** (Bollerslev, 1986):

$$\sigma_t^2 = \omega + \sum_{i=1}^{q}\alpha_i r_{t-i}^2 + \sum_{j=1}^{p}\beta_j \sigma_{t-j}^2$$

**Most common: GARCH(1,1):**
$$\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$$

**Conditions:**
- $\omega > 0$, $\alpha \geq 0$, $\beta \geq 0$
- $\alpha + \beta < 1$ (covariance stationarity)
- $\alpha + \beta \approx 1$ => **IGARCH** (volatility shocks are persistent)


In [ ]:
# ── GARCH(1,1) Model Estimation ──

# arch package expects returns scaled by 100
r_scaled = r * 100

garch_model = arch_model(r_scaled, vol='Garch', p=1, q=1, mean='Constant', dist='Normal')
garch_fit = garch_model.fit(disp='off')

print(garch_fit.summary())
print(f"\nalpha + beta = {garch_fit.params['alpha[1]'] + garch_fit.params['beta[1]']:.4f}")
print("(Close to 1: high volatility persistence)")


In [ ]:
# ── GARCH(1,1): Estimated vs True Volatility ──

cond_vol = garch_fit.conditional_volatility / 100  # Rescale

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(r, color='gray', lw=0.5, alpha=0.6, label='Returns')
axes[0].plot(2*cond_vol, color='red', lw=1.2, alpha=0.8, label='+2*sigma')
axes[0].plot(-2*cond_vol, color='red', lw=1.2, alpha=0.8, label='-2*sigma')
axes[0].set_title('Log Returns and GARCH(1,1) Volatility Band', fontweight='bold')
axes[0].legend()

axes[1].plot(sigma_t, color='steelblue', lw=1.2, label='True sigma_t', alpha=0.8)
axes[1].plot(cond_vol, color='red', lw=1, ls='--', label='GARCH Estimated sigma_t', alpha=0.8)
axes[1].set_title('True vs Estimated Volatility', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

n_min = min(len(sigma_t), len(cond_vol))
corr = np.corrcoef(sigma_t[:n_min], cond_vol[:n_min])[0,1]
print(f"Correlation between true and estimated volatility: {corr:.3f}")


## 7.4 EGARCH — Asymmetric Volatility

**EGARCH(1,1)** (Nelson, 1991):

$$\log\sigma_t^2 = \omega + \alpha\left(\frac{|\epsilon_{t-1}|}{\sigma_{t-1}} - \sqrt{2/\pi}\right) + \gamma\frac{\epsilon_{t-1}}{\sigma_{t-1}} + \beta\log\sigma_{t-1}^2$$

**Leverage effect:** If $\gamma < 0$, negative shocks increase volatility more than positive shocks (typical in equity markets).


In [ ]:
# ── EGARCH Leverage Effect ──

egarch_model = arch_model(r_scaled, vol='EGARCH', p=1, q=1, mean='Constant')
egarch_fit = egarch_model.fit(disp='off')

print("EGARCH Parameters:")
for p, v, pv in zip(egarch_fit.params.index,
                     egarch_fit.params.values,
                     egarch_fit.pvalues.values):
    sig = "***" if pv < 0.01 else ("**" if pv < 0.05 else ("*" if pv < 0.1 else ""))
    print(f"  {p:15s}: {v:8.5f}  (p={pv:.4f}) {sig}")

gamma = egarch_fit.params.get('gamma[1]', egarch_fit.params.get('gamma', None))
if gamma is not None:
    print(f"\nLeverage effect (gamma): {gamma:.4f}")
    print(f"{'Negative => Leverage effect PRESENT (typical equity behaviour)' if gamma<0 else 'Positive => Reverse leverage'}")


## 7.5 Geometric Brownian Motion (GBM) and Black-Scholes

**GBM Stochastic Differential Equation:**
$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

**Solution:**
$$S_t = S_0 \exp\left[(\mu - \tfrac{\sigma^2}{2})t + \sigma W_t\right]$$

**Black-Scholes Call Option Price:**
$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$


In [ ]:
# ── GBM Simulation + Black-Scholes ──

from scipy.stats import norm

def gbm_sim(S0, mu, sigma, T, n_steps, n_paths=5, seed=42):
    np.random.seed(seed)
    dt = T / n_steps
    t = np.linspace(0, T, n_steps+1)
    paths = np.zeros((n_paths, n_steps+1))
    paths[:, 0] = S0
    for i in range(1, n_steps+1):
        Z = np.random.normal(0, 1, n_paths)
        paths[:, i] = paths[:, i-1] * np.exp((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
    return t, paths

def black_scholes_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2), d1, d2

# Parameters
S0, mu_gbm, sigma_gbm, T_year = 100, 0.10, 0.25, 1.0
t, paths = gbm_sim(S0, mu_gbm, sigma_gbm, T_year, n_steps=252, n_paths=20)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# GBM paths
for i in range(20):
    axes[0].plot(t, paths[i], lw=0.7, alpha=0.6)
axes[0].axhline(S0, color='black', ls='--', label=f'S0={S0}')
axes[0].set_title(f'GBM Simulation (mu={mu_gbm}, sigma={sigma_gbm})', fontweight='bold')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Price')
axes[0].legend()

# Black-Scholes: option value for different K
K_vals = np.linspace(70, 130, 50)
r_rf = 0.05
C_vals = [black_scholes_call(S0, K, r_rf, sigma_gbm, T_year)[0] for K in K_vals]
axes[1].plot(K_vals, C_vals, color='steelblue', lw=2)
axes[1].axvline(S0, color='red', ls='--', label=f'S0={S0} (ATM)')
axes[1].set_title('Black-Scholes Call Option Value', fontweight='bold')
axes[1].set_xlabel('Strike Price K')
axes[1].set_ylabel('Option Value C')
axes[1].legend()

plt.tight_layout()
plt.show()

# Example computation
C, d1, d2 = black_scholes_call(S0, 100, r_rf, sigma_gbm, T_year)
print(f"ATM Call Option (S0=K=100): C = {C:.4f}")
print(f"d1={d1:.4f}, d2={d2:.4f}")
print(f"Delta = N(d1) = {norm.cdf(d1):.4f}")


## Summary

| Model | Use |
|-------|-----|
| **ARCH(q)** | Basic conditional heteroskedasticity |
| **GARCH(1,1)** | Standard volatility model |
| **EGARCH** | Asymmetric/leverage effect |
| **GBM** | Continuous-time price model |
| **Black-Scholes** | Option pricing |

**Key concepts:**
- Volatility clustering => ARCH/GARCH
- $\alpha + \beta \to 1$ => volatility shocks are long-lived
- Leverage effect ($\gamma < 0$) => EGARCH
